In [32]:
from multiprocessing import resource_tracker

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Sequence
from IPython.display import display
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from langchain.tools import tool
from loguru import logger
from langgraph.types import Send, Command
from langchain.messages import HumanMessage, AIMessage, SystemMessage
load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    # temperature=0.7,
    extra_body={
        "thinking": {
            "type": "disabled",
        }
    }
)
@tool(parse_docstring=True)
def get_weather(city: str = "上海"):
    """
    查询指定城市的天气

    Args:
        city: 城市名称
    """
    return f"{city}的天气是晴朗的。"

@tool(parse_docstring=True)
def get_news(domain:Literal["AI","食品安全"]):
    """
    查询新闻

    Args:
        domain: 查询新闻
    """
    if domain == "AI":
        return "AI is developed in very quick speed..."
    elif domain == "食品安全":
        return "The food security situation is more and more serious..."
    else:
        return "Unknow news area"


#绑定工具到大模型
tools = [get_weather, get_news]

model_with_tool = model.bind_tools(tools=tools)
res = model_with_tool.invoke("上海的天气怎么样？")
print(res)


content='我来帮您查询上海的天气情况。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 343, 'total_tokens': 395, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 256, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 87}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': 'a54d9fb4-8496-4d78-83ae-6169c1b1aa82', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a05374-8b52-7111-8b76-0e7a41337f97-0' tool_calls=[{'name': 'get_weather', 'args': {'city': '上海'}, 'id': 'call_00_ZftioWyaGOoa7WIYkj4H5879', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 343, 'output_tokens': 52, 'total_tokens': 395, 'input_token_details': {'cache_read': 256}, 'output_token_details': {}}


In [33]:
# from langchain import messages
from langchain_core.messages import ToolMessage, SystemMessage
from random import randint
from langgraph.graph import MessagesState


#输入节点，将用户输入的查询信息，记录到message中，方便后续的大模型调用
class OverAllState(MessagesState):
    user_input: str
    final_output: str

def input_node(state: OverAllState):
    return {
        "message": [HumanMessage(state["user_input"])]
    }

def llm_node(state: OverAllState):
    ai_msg = model_with_tool.invoke(state["messages"])
    return {
        "message": ai_msg
    }

#判断是否需要调用tool
def tool_node(state: OverAllState) -> OverAllState:
    messages = state["messages"]
    ai_msg = messages[-1]
    tool_calls = ai_msg.tool_calls
    fail_prob = 6 # 模拟工具调用失败概率
    for tool_call in tool_calls:
        if tool_call["name"] == "get_weather":# 生成一个0-9数字判断大小来模拟
            tempNumber1 = randint(0,9)
            if tempNumber1 < fail_prob:
                print("tempNumber1 is: ", tempNumber1 ,"\n")
                messages.append(ToolMessage(
                    content="网络波动，调用失败，请重试",
                    tool_call_id = tool_call["id"]
                ))
            else:
                messages.append(get_weather.invoke(tool_call))

        elif tool_call["name"] == "get_news":# 生成一个0-9数字判断大小来模拟
            tempNumber2 = randint(0,9)
            if tempNumber2 < fail_prob:
                print("tempNumber2 is: ", tempNumber2 ,"\n")
                messages.append(ToolMessage(
                    content="网络波动，调用失败，请重试",
                    tool_call_id = tool_call["id"]
                ))
            else:
                messages.append(get_news.invoke(tool_call))
        else:
             messages.append(ToolMessage(content="工具名称错误。"))

    return {
        "messages": messages
    }

#返回输出节点
def output_node(state: OverAllState) -> OverAllState:
    return {
        "final_output": state["messages"][-1].content

    }
def router(state:OverAllState) -> Literal["tool_node", "output_node"]:
    messages = state["messages"]
    last_msg = messages[-1]
    if last_msg.tool_calls:
        return "tool_node"
    return "output_node"
builder = StateGraph(state_schema=OverAllState)
builder.add_node("input_node", input_node)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_node("output_node", output_node)
builder.add_edge(START, "input_node")
builder.add_edge("input_node", "llm_node")
builder.add_conditional_edges("llm_node", router)
builder.add_edge("tool_node", "llm_node")
builder.add_edge("output_node", END)

graph = builder.compile()

res = graph.invoke({
    "user_input": "查询今天上海天气和AI新闻热点",
    "messages": [SystemMessage("如果工具调用失败，必须重新调用到成功为止")]
})
print("user_inputer: ", res["user_input"])
print("final_output: ", res["final_output"])
for msg in res["messages"]:
    msg.pretty_print()
display(graph)



AttributeError: 'SystemMessage' object has no attribute 'tool_calls'